In [ ]:
import json

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import gaussian_kde

from analysis.src.io import load_data, read_table
from analysis.src.plotting import configure_plotting
from analysis.src.constants import (
    OUTCOME_LABELS, OUTCOME_ORDER, RADIAL_BAND_ORDER, RADIAL_BREAKPOINTS_KM,
    SECTOR_ORDER, PRIMARY_ADJUSTMENT
)
from paths import paths

In [ ]:
INK, NAVY, BLUE, TEAL = '#1C2730', '#17324D', '#2F6B8A', '#3E8C8A'
GOLD, TERRACOTTA, PLUM = '#C39A43', '#C66B4E', '#7B637D'
PALE, LIGHT_GREY, MID_GREY, DARK_GREY, GRID, WHITE = ('#E4EAEB', '#DCE4E7', '#8A969C', '#56636A', '#D8E0E3', '#FFFFFF')

WALKABILITY_CMAP = LinearSegmentedColormap.from_list(
    'walkability', ['#F2F6F6', '#C8DEDC', '#78B6B0', TEAL, BLUE, NAVY],
)
SHORTFALL_COLOURS = {
    'Amenity access': TEAL,
    'Dwelling density': BLUE,
    'Land-use mix': GOLD,
    'Street connectivity': TERRACOTTA,
    'Transport access': PLUM,
    'Mixed': MID_GREY
}

TITLE_X, TITLE_Y, TITLE_SIZE = 0.0, 0.985, 10.5
PANEL_WSPACE, PANEL_HSPACE = 0.0625, 0.10
XLABEL_PAD = 10

OUTCOME_POSITIONS = np.arange(len(OUTCOME_ORDER))
SPATIAL_LIMITS = (len(OUTCOME_ORDER) - 0.45, -0.55)

In [ ]:
configure_plotting()

mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 9,
    'axes.titlesize': 9.5, 'axes.labelsize': 9, 'axes.titleweight': 'semibold',
    'axes.labelcolor': INK, 'axes.edgecolor': DARK_GREY, 'axes.linewidth': 0.75,
    'xtick.color': DARK_GREY, 'ytick.color': DARK_GREY,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'text.color': INK,
    'legend.fontsize': 9, 'legend.frameon': False,
    'figure.facecolor': WHITE, 'axes.facecolor': WHITE, 'savefig.facecolor': WHITE,
    'savefig.transparent': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
})

In [ ]:
def clean_axis(ax, grid_axis = None):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color(LIGHT_GREY)

    ax.tick_params(length = 2.8, width = 0.65, color = MID_GREY)

    if grid_axis:
        ax.grid(axis = grid_axis, color = GRID, linewidth = 0.65, alpha = 0.7)
        ax.set_axisbelow(True)

def inset(panel, left, right, top, height):
    return panel.inset_axes([left, top - height, 1 - left - right, height])

def fit_map(ax, gdf, pad = 0.012):
    xmin, ymin, xmax, ymax = gdf.total_bounds
    dx, dy = xmax - xmin, ymax - ymin

    ax.set_xlim(xmin - dx * pad, xmax + dx * pad)
    ax.set_ylim(ymin - dy * pad, ymax + dy * pad)
    ax.set_aspect('equal')
    ax.set_anchor('C')
    ax.set_axis_off()

def save_figure(fig, stem):
    fig.savefig(OUT / f'{stem}.pdf', dpi = 600, bbox_inches = 'tight')
    fig.savefig(OUT / f'{stem}.svg', bbox_inches = 'tight')
    fig.savefig(OUT / f'{stem}.png', dpi = 600, bbox_inches = 'tight')

In [ ]:
def panel_grid(titles, shape, figsize, *, wspace = PANEL_WSPACE, hspace = PANEL_HSPACE, height_ratios = None):
    fig = plt.figure(figsize = figsize)
    gs = fig.add_gridspec(*shape, wspace = wspace, hspace = hspace, height_ratios = height_ratios)
    panels = []

    for title, cell in zip(titles, gs):
        ax = fig.add_subplot(cell)
        ax.set_xticks([])
        ax.set_yticks([])

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.patch.set_alpha(0)
        ax.text(
            TITLE_X, TITLE_Y, title, transform = ax.transAxes, ha = 'left', va = 'top', fontsize = TITLE_SIZE,
            fontweight = 'semibold', color = INK,
        )
        panels.append(ax)

    return fig, panels

def panel_legend(panel, handles, *, y = None, ncol = 2, fontsize = 9, x = TITLE_X, **kwargs):
    defaults = {
        'loc': 'upper left',
        'bbox_to_anchor': (x, TITLE_Y - 0.05 if y is None else y),
        'ncol': ncol,
        'frameon': False,
        'fontsize': fontsize,
        'borderpad': 0,
        'borderaxespad': 0,
        'handlelength': 0.75,
        'handletextpad': 0.4,
        'columnspacing': 1.2
    }
    defaults.update(kwargs)

    return panel.legend(handles = handles, **defaults)

def marker_handle(label, colour, marker = 'o', size = 6, **kwargs):
    properties = dict(
        marker = marker, linestyle = 'none', markersize = size,
        markerfacecolor = colour, markeredgecolor = 'none', label = label,
    )
    properties.update(kwargs)

    return Line2D([0], [0], **properties)

In [ ]:
OUT = paths.results.figures
OUT.mkdir(parents = True, exist_ok = True)

data = load_data()
walkability_sd = data.walkability_score.std(ddof = 1)
inputs = paths.analysis.experiments

In [ ]:
# Neighbourhood characteristics and spatial summaries
geography = inputs['geography/gradients_and_shortfalls']

spatial = gpd.read_parquet(geography.spatial_parquet)
boroughs = gpd.read_parquet(geography.boroughs_parquet)
geographic_summary = json.loads(geography.summary_json.read_text())

shortfalls = spatial.loc[spatial.is_shortfall].copy()
shortfall_summary = shortfalls.dominant_shortfall.value_counts().reindex(SHORTFALL_COLOURS).fillna(0).astype(int)

radial_curve = read_table(geography.radial_curve_csv)
radial_segments = read_table(geography.radial_segments_csv)
sector_band = read_table(geography.sector_band_csv).set_index('sector')

In [ ]:
def draw_walkability(panel):
    ax = panel.inset_axes([0, 0.10, 1, 0.80])
    
    vmin, vmax = np.nanpercentile(spatial['walkability_score'], [2.5, 97.5])
    edges = np.arange(np.floor(vmin / 0.25) * 0.25, np.ceil(vmax / 0.25) * 0.25 + 0.25, 0.25)
    cmap = ListedColormap(WALKABILITY_CMAP(np.linspace(0.10, 1, len(edges) - 1)))
    norm = BoundaryNorm(edges, cmap.N, clip = True)
    
    spatial.plot(ax = ax, column = 'walkability_score', cmap = cmap, norm = norm, linewidth = 0)

    if boroughs is not None:
        boroughs.boundary.plot(ax = ax, color = WHITE, linewidth = 0.35, alpha = 0.85)

    fit_map(ax, spatial)

    cax = panel.inset_axes([0, 0.035, 0.96, 0.038])
    mappable = mpl.cm.ScalarMappable(norm = norm, cmap = cmap)
    mappable.set_array([])
    ticks = np.arange(edges[0], edges[-1] + 1e-9, 0.5)

    if ticks[-1] != edges[-1]:
        ticks = np.append(ticks, edges[-1])

    colourbar = panel.figure.colorbar(
        mappable, cax = cax, orientation = 'horizontal', boundaries = edges, ticks = ticks,
        spacing = 'proportional',
    )
    colourbar.outline.set_visible(False)
    colourbar.ax.tick_params(labelsize = 7, length = 2)
    colourbar.set_label('Composite score', fontsize = 9, labelpad = 5)

In [ ]:
def draw_shortfalls(panel):
    ax = panel.inset_axes([0, 0.10, 1, 0.80])
    spatial.plot(ax = ax, color = PALE, linewidth = 0)

    for label, colour in SHORTFALL_COLOURS.items():
        shortfalls.loc[shortfalls['dominant_shortfall'] == label].plot(ax = ax, color = colour, linewidth = 0)

    if boroughs is not None:
        boroughs.boundary.plot(ax = ax, color = WHITE, linewidth = 0.35, alpha = 0.85)

    fit_map(ax, spatial)
    panel.text(
        0, TITLE_Y - 0.05, f'{len(shortfalls):,} LSOAs ({len(shortfalls) / len(spatial):.1%})',
        transform = panel.transAxes, ha = 'left', va = 'top', fontsize = 9, color = DARK_GREY,
    )
    handles = [
        Patch(facecolor = colour, edgecolor = 'none', label = f'{label} · {shortfall_summary[label]:,}')
        for label, colour in SHORTFALL_COLOURS.items()
    ]
    panel.legend(
        handles = handles, loc = 'lower left', bbox_to_anchor = (0, 0), ncol = 3, fontsize = 8.5, handlelength = 0.75,
        handletextpad = 0.45, columnspacing = 1, labelspacing = 0.4, borderaxespad = 0, frameon = False,
    )

In [ ]:
def draw_gradient(panel):
    ax = inset(panel, 0.10, 0.02, 0.90, 0.82)
    ax.scatter(
        spatial['distance_km'], spatial['walkability_score'], s = 5.5, alpha = 0.15, color = TEAL, edgecolors = 'none',
        rasterized = True,
    )
    ax.plot(
        radial_curve['distance_km'], radial_curve['predicted'], color = NAVY, linewidth = 2, linestyle = (0, (0.1, 2)),
        dash_capstyle = 'round', zorder = 4,
    )

    for x in RADIAL_BREAKPOINTS_KM:
        ax.axvline(x, color = MID_GREY, linewidth = 0.85, linestyle = (0, (3, 3)))

    ax.axhline(0, color = LIGHT_GREY, linewidth = 0.7)
    ax.set(xlabel = 'Distance from central London (km)', ylabel = 'Composite walkability score', ylim = (-1.45, 2.55))
    ax.xaxis.labelpad = XLABEL_PAD
    
    clean_axis(ax, 'y')
    
    ax.text(
        0.855, 0.82, f'$R^2$ = {geographic_summary["radial_r_squared"]:.3f}', transform = ax.transAxes, ha = 'right',
        va = 'top', fontsize = 9, fontweight = 'semibold', color = NAVY,
    )

    for segment in radial_segments.itertuples():
        ax.text(
            (segment.start_km + segment.end_km) / 2, 2.4, f'{segment.band}\n{segment.slope_sd_per_km:.3f} SD/km',
            ha = 'center', va = 'top', fontsize = 9, fontweight = 'medium', color = INK, linespacing = 1.15,
        )

In [ ]:
def draw_sectors(panel):
    ax = inset(panel, 0.0625, 0.02, 0.88, 0.80)
    band_colours = dict(zip(RADIAL_BAND_ORDER, [NAVY, TEAL, GOLD]))

    for i, sector in enumerate(SECTOR_ORDER):
        values = sector_band.loc[sector].to_numpy()
        ax.plot(values, np.full(3, i), color = LIGHT_GREY, linewidth = 1.5)

        for value, band in zip(values, RADIAL_BAND_ORDER):
            ax.scatter(value, i, s = 34, color = band_colours[band], edgecolor = WHITE, linewidth = 0.4, zorder = 3)

    ax.set_yticks(range(8), SECTOR_ORDER)
    ax.invert_yaxis()
    ax.set(xlabel = 'Mean walkability score', xlim = (-1.15, 1.28))
    ax.xaxis.labelpad = XLABEL_PAD
    
    clean_axis(ax, 'x')
    
    band_handles = [
        marker_handle(band, band_colours[band], markeredgecolor = WHITE, markeredgewidth = 0.4)
        for band in RADIAL_BAND_ORDER
    ]
    
    panel_legend(panel, band_handles, ncol = 3, handlelength = 0.75, handletextpad = 0.35, columnspacing = 1.15)

In [ ]:
# Figure 1 – spatial anatomy
fig, panels = panel_grid(
    [
        'A: Geography of Walkability',
        'B: Local Shortfalls Relative to Radial Expectation',
        'C: Centre–Periphery Gradient',
        'D: Directional Variation by Radial Band',
    ],
    (2, 2), (10, 9), height_ratios = [1.1, 1],
)

draw_walkability(panels[0])
draw_shortfalls(panels[1])
draw_gradient(panels[2])
draw_sectors(panels[3])

fig.subplots_adjust(left = 0.04, right = 0.995, top = 0.98, bottom = 0.04)

save_figure(fig, 'spatial_anatomy')

plt.show()

In [ ]:
# Association estimates and plotting units
sequential = read_table(inputs['associations/primary'].sequential_models_csv)
quintile_results = read_table(inputs['associations/dose_response'].quintile_contrasts_csv)

HEALTH_MODELS = {
    'Primary': ('Primary', NAVY, 'o', 5.8, -0.16),
    'Primary + NOx': ('+ NOx', GOLD, 's', 5.0, 0.00),
    'Primary + NDVI': ('+ NDVI', TEAL, '^', 5.0, 0.16)
}
health = sequential.loc[sequential['model'].isin(HEALTH_MODELS)].copy()

for c in ['coefficient', 'ci_lower', 'ci_upper']:
    health[f'{c}_pp'] = health[c] * walkability_sd * 100

quintiles = quintile_results.copy()
quintiles['difference_pp'] = quintiles['coefficient'] * 100
QUINTILES = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
quintile_matrix = quintiles.pivot(index = 'outcome', columns = 'quintile', values = 'difference_pp').reindex(OUTCOME_ORDER)[
    ['Q2', 'Q3', 'Q4', 'Q5']
]
quintile_matrix.insert(0, 'Q1', 0.0)

In [ ]:
def draw_associations(panel, ax):
    for position in OUTCOME_POSITIONS:
        ax.axhline(position, color = '#F0F3F4', linewidth = 0.7)

    for model, (label, colour, marker, size, offset) in HEALTH_MODELS.items():
        estimates = health.loc[health['model'] == model].set_index('outcome').reindex(OUTCOME_ORDER)
        estimate, lower, upper = (estimates[column].to_numpy() for column in ['coefficient_pp', 'ci_lower_pp', 'ci_upper_pp'])
        ax.errorbar(
            estimate, OUTCOME_POSITIONS + offset, xerr = np.vstack([estimate - lower, upper - estimate]),
            fmt = marker, color = colour, ecolor = colour, markersize = size,
            elinewidth = 1.2 if model == 'Primary' else 0.95, capsize = 2.2, markeredgewidth = 0,
            zorder = 5 if model == 'Primary' else 4,
        )

    ax.axvline(0, color = INK, linewidth = 0.85)
    ax.set_yticks(OUTCOME_POSITIONS, [OUTCOME_LABELS[outcome] for outcome in OUTCOME_ORDER])
    ax.set_ylim(len(OUTCOME_POSITIONS) - 0.55, -0.55)
    ax.set_xlabel('Difference in prevalence (pp per 1-SD higher walkability)', labelpad = XLABEL_PAD)
    
    xmin = health['ci_lower_pp'].min() * 1.10
    ax.set_xlim(xmin, max(0.08, abs(xmin) * 0.055))
    
    clean_axis(ax, 'x')
    
    panel_legend(
        panel, [marker_handle(style[0], style[1], style[2], style[3]) for style in HEALTH_MODELS.values()],
        ncol = 3, columnspacing = 1.1,
    )

In [ ]:
def draw_quintiles(panel, ax):
    trajectory_colours = {
        'Hypertension': NAVY,
        'Obesity': GOLD,
        'Diabetes': TEAL,
        'CHD': TERRACOTTA,
        'AF': '#8E66CC',
        'Stroke / TIA': '#8B5E4A',
        'HF': '#E170C8',
        'PAD': '#7A7F85'
    }
    
    x = np.arange(1, 6)

    for outcome in OUTCOME_ORDER:
        label = OUTCOME_LABELS[outcome]
        ax.plot(
            x, quintile_matrix.loc[outcome, QUINTILES], marker = 'o', linewidth = 1, markersize = 4, label = label,
            color = trajectory_colours[label], zorder = 3,
        )

    ax.axhline(0, color = INK, linewidth = 0.85)
    ax.set_xticks(x, QUINTILES)
    ax.set_xlabel('Walkability quintile', labelpad = XLABEL_PAD)
    ax.set_ylabel('Adjusted prevalence difference vs Q1 (pp)', labelpad = 7.5)
    ax.set_ylim(-4, 0)
    ax.grid(axis = 'y', linewidth = 0.6, alpha = 0.5)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(
        loc = 'lower left', bbox_to_anchor = (0.03, 0.015), ncol = 2, frameon = False, fontsize = 9,
        handlelength = 0.75, columnspacing = 1.2, borderaxespad = 0,
    )
    
    panel.text(
        0, TITLE_Y - 0.05, 'Adjusted prevalence difference versus Q1', transform = panel.transAxes, ha = 'left',
        va = 'top', fontsize = 9, color = DARK_GREY,
    )
    
    clean_axis(ax, 'y')

In [ ]:
# Figure 2 – associations and dose-response
fig, panels = panel_grid(
    ['A: Adjusted Associations and Environmental Sensitivity', 'B: Dose–Response Across Walkability Quintiles'],
    (1, 2), (10, 4.9), wspace = 0.025,
)
association_ax = inset(panels[0], 0.18, 0.03, 0.885, 0.80)
quintile_ax = inset(panels[1], 0.12, 0.02, 0.885, 0.80)

draw_associations(panels[0], association_ax)
draw_quintiles(panels[1], quintile_ax)

fig.subplots_adjust(left = 0.035, right = 0.995, top = 0.98, bottom = 0.06)

save_figure(fig, 'health_associations')

plt.show()

In [ ]:
# Spatial model, null, and cross-validation results
borough_results = read_table(inputs['spatial/borough_fixed_effects'].borough_models_csv)
spatial_results = read_table(inputs['spatial/autocorrelation_and_error_models'].spatial_error_models_csv)
null_draws = read_table(inputs['spatial/conditional_null'].conditional_null_draws_csv)
null_results = read_table(inputs['spatial/conditional_null'].conditional_null_summary_csv)
cv_results = read_table(inputs['prediction/spatial_cross_validation'].spatial_cross_validation_csv)

borough = borough_results.query('adjustment == @PRIMARY_ADJUSTMENT')

def borough_pivot(col):
    return borough.pivot(index = 'outcome', columns = 'model', values = col).reindex(OUTCOME_ORDER) * 100

borough_est = borough_pivot('coefficient_per_sd')
borough_lo = borough_pivot('ci_lower_per_sd')
borough_hi = borough_pivot('ci_upper_per_sd')

spatial_error = (
    spatial_results.query('adjustment == @PRIMARY_ADJUSTMENT').set_index('outcome').reindex(OUTCOME_ORDER)
)
spatial_error['conventional_pct'] = np.expm1(spatial_error['glm_coefficient'] * walkability_sd) * 100
spatial_error['spatial_pct'] = np.expm1(spatial_error['spatial_error_coefficient'] * walkability_sd) * 100

null_all = null_draws.query('adjustment == @PRIMARY_ADJUSTMENT')
null_summary = null_results.query('adjustment == @PRIMARY_ADJUSTMENT').set_index('outcome')
null_z, null_obs, null_q = {}, {}, {}

for outcome in OUTCOME_ORDER:
    values = null_all.loc[null_all['outcome'] == outcome, 'coefficient_per_sd'].dropna()
    median, sd = values.median(), values.std(ddof = 1)
    null_z[outcome] = ((values - median) / sd).to_numpy()
    null_obs[outcome] = (null_summary.loc[outcome, 'observed_coefficient_per_sd'] - median) / sd
    null_q[outcome] = null_summary.loc[outcome, 'empirical_q_value']

spatial_cv = (
    cv_results.drop_duplicates(['outcome', 'fold'])[['outcome', 'fold', 'incremental_r_squared']].copy()
)
cv_summary = spatial_cv.groupby('outcome')['incremental_r_squared'].agg(['mean', 'min', 'max']).reindex(OUTCOME_ORDER)

In [ ]:
def draw_boroughs(panel, ax):
    for i, outcome in enumerate(OUTCOME_ORDER):
        ax.plot(
            [borough_est.loc[outcome, 'Adjusted'], borough_est.loc[outcome, 'Borough fixed effects']], [i, i],
            color = LIGHT_GREY, linewidth = 1.5,
        )

    for model, colour, marker, size in [('Adjusted', NAVY, 'o', 34), ('Borough fixed effects', GOLD, 's', 30)]:
        estimate, lower, upper = (x[model].to_numpy() for x in [borough_est, borough_lo, borough_hi])
        ax.errorbar(
            estimate, OUTCOME_POSITIONS, xerr = np.vstack([estimate - lower, upper - estimate]), fmt = marker,
            color = colour, ecolor = colour, markersize = np.sqrt(size), elinewidth = 1.05, capsize = 2,
            markeredgewidth = 0, zorder = 3,
        )

    ax.axvline(0, color = INK, linewidth = 0.8)
    ax.set_ylim(*SPATIAL_LIMITS)
    ax.set_yticks(OUTCOME_POSITIONS, [OUTCOME_LABELS[outcome] for outcome in OUTCOME_ORDER])
    ax.set_xlabel('Difference in prevalence (pp per SD)', labelpad = XLABEL_PAD)
    
    clean_axis(ax, 'x')
    
    panel_legend(
        panel, [marker_handle('Primary adjusted', NAVY), marker_handle('Borough fixed effects', GOLD, 's', 5.5)],
        ncol = 2,
    )

In [ ]:
def draw_spatial_error(panel, ax):
    for i, outcome in enumerate(OUTCOME_ORDER):
        ax.plot(
            [spatial_error.loc[outcome, 'conventional_pct'], spatial_error.loc[outcome, 'spatial_pct']], [i, i],
            color = LIGHT_GREY, linewidth = 1.5,
        )

    ax.scatter(
        spatial_error['conventional_pct'], OUTCOME_POSITIONS, s = 34, color = NAVY, edgecolor = 'none', zorder = 3,
    )
    ax.scatter(
        spatial_error['spatial_pct'], OUTCOME_POSITIONS, s = 30, color = GOLD, marker = 's', edgecolor = 'none',
        zorder = 3,
    )
    ax.axvline(0, color = INK, linewidth = 0.8)
    ax.set_ylim(*SPATIAL_LIMITS)
    ax.set_yticks(OUTCOME_POSITIONS, [])
    ax.set_xlabel('Relative difference in prevalence (% per SD)', labelpad = XLABEL_PAD)
    
    clean_axis(ax, 'x')

    for i, outcome in enumerate(OUTCOME_ORDER):
        attenuation = min(
            100,
            100 * (1 - spatial_error.loc[outcome, 'spatial_error_coefficient'] / spatial_error.loc[outcome, 'glm_coefficient']),
        )
        ax.text(
            1.015, i, f'{attenuation:.0f}%', transform = ax.get_yaxis_transform(), ha = 'left', va = 'center',
            fontsize = 9, color = DARK_GREY, clip_on = False,
        )

    ax.text(1, 1.015, 'atten.', transform = ax.transAxes, ha = 'left', va = 'bottom', fontsize = 9, color = MID_GREY)
    
    panel_legend(panel, [marker_handle('Conventional', NAVY), marker_handle('Spatial-error', GOLD, 's', 5.5)], ncol = 2)

In [ ]:
def draw_spatial_null(panel, ax):
    xgrid = np.linspace(-7.6, 3.6, 500)

    for i, outcome in enumerate(OUTCOME_ORDER):
        density = gaussian_kde(null_z[outcome])(xgrid)
        density = density / density.max() * 0.28
        ax.fill_between(xgrid, i - density, i + density, color = TEAL, alpha = 0.22, linewidth = 0)
        ax.plot(xgrid, i + density, color = TEAL, linewidth = 0.8)
        ax.plot(xgrid, i - density, color = TEAL, linewidth = 0.8)
        ax.scatter(
            null_obs[outcome], i, s = 28, marker = 'D', color = TERRACOTTA, edgecolor = WHITE, linewidth = 0.4,
            zorder = 5,
        )
        
        q = null_q[outcome]
        q_label = f'q = {q:.3f}' if q >= 0.001 else 'q < .001'
        ax.text(3.48, i - 0.225, q_label, ha = 'right', fontsize = 9, color = DARK_GREY)

    ax.axvline(0, color = MID_GREY, linewidth = 0.75, linestyle = (0, (3, 3)))
    ax.set(xlim = (-7.6, 3.6), ylim = SPATIAL_LIMITS)
    ax.set_yticks(OUTCOME_POSITIONS, [OUTCOME_LABELS[outcome] for outcome in OUTCOME_ORDER])
    ax.set_xlabel('Standard deviations from spatially constrained null median', labelpad = XLABEL_PAD)
    
    clean_axis(ax)
    
    panel_legend(
        panel,
        [
            Patch(facecolor = TEAL, edgecolor = TEAL, alpha = 0.22, label = 'Conditional null'),
            marker_handle('Observed coefficient', TERRACOTTA, 'D', 5.5, markeredgecolor = WHITE, markeredgewidth = 0.4)
        ],
        ncol = 2,
        handlelength = 0.75,
        handletextpad = 0.25,
    )

In [ ]:
def draw_cross_validation(panel, ax):
    cv_min, cv_max = spatial_cv['incremental_r_squared'].agg(['min', 'max'])
    span = max(cv_max - cv_min, 0.01)
    x_min = min(0, cv_min - 0.05 * span)
    x_max = cv_max + 0.10 * span
    
    ax.axvline(0, color = INK, linewidth = 0.8)

    for i, outcome in enumerate(OUTCOME_ORDER):
        summary = cv_summary.loc[outcome]
        folds = spatial_cv.loc[spatial_cv['outcome'] == outcome, 'incremental_r_squared'].to_numpy()
        ax.plot(
            [summary['min'], summary['max']], [i, i], color = LIGHT_GREY, linewidth = 1.5,
            solid_capstyle = 'round',
        )
        ax.scatter(
            folds, np.full_like(folds, i), s = 18, color = TEAL, edgecolor = WHITE, linewidth = 0.35, zorder = 2,
        )
        ax.scatter(summary['mean'], i, s = 42, color = NAVY, edgecolor = WHITE, linewidth = 0.45, zorder = 3)
        ax.text(x_max, i, f'{summary["mean"]:.3f}', ha = 'left', va = 'center', fontsize = 9, color = DARK_GREY)

    ax.set(xlim = (x_min, x_max), ylim = SPATIAL_LIMITS)
    ax.set_yticks(OUTCOME_POSITIONS, [])
    ax.set_xlabel('Incremental held-out $R^2$', labelpad = XLABEL_PAD)
    
    clean_axis(ax, 'x')
    
    panel_legend(panel, [marker_handle('Mean Δ$R^2$', NAVY), marker_handle('Spatial folds', TEAL, size = 5)], ncol = 2)

In [ ]:
# Figure 3 – spatial robustness and out-of-area prediction
fig, panels = panel_grid(
    [
        'A: Within-Borough Comparison',
        'B: Explicit Spatial Dependence',
        'C: Spatially Constrained Null',
        'D: Geographic Out-of-Sample Prediction',
    ],
    (2, 2), (10, 8),
)

borough_ax = inset(panels[0], 0.18, 0, 0.885, 0.80)
null_ax = inset(panels[2], 0.18, 0, 0.885, 0.80)
error_ax = inset(panels[1], 0.02, 0.03, 0.885, 0.80)
cv_ax = inset(panels[3], 0.02, 0.03, 0.885, 0.80)

draw_boroughs(panels[0], borough_ax)
draw_spatial_error(panels[1], error_ax)
draw_spatial_null(panels[2], null_ax)
draw_cross_validation(panels[3], cv_ax)

fig.subplots_adjust(left = 0.04, right = 0.99, top = 0.98, bottom = 0.04)

save_figure(fig, 'spatial_dependence')

plt.show()